# Standard phase retrieval

Minimal two-helicity workflow using `library/phase_retrieval_core.py`. Centering, support construction, propagation, and experiment-specific preprocessing are intentionally left out.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from library import phase_retrieval_core as phr

## Load prepared arrays

The input file should contain 2D arrays named `pos`, `neg`, `mask_pixel`, and `supportmask`. `pos` and `neg` are measured intensities. Nonzero `mask_pixel` values mark unconstrained Fourier pixels.

In [ ]:
data = np.load(Path("data/phase_retrieval_inputs.npz"))
pos = data["pos"]
neg = data["neg"]
mask_pixel = data["mask_pixel"]
supportmask = data["supportmask"]

assert pos.shape == neg.shape == mask_pixel.shape == supportmask.shape

## Reconstruction recipe

This compact recipe runs HAPRE followed by ER for both helicities. Increase the iteration counts for production reconstruction.

In [ ]:
recipe = {
    "algorithm_list": ["HAPRE", "ER", "ER"],
    "number_iterations": [500, 100, 100],
    "helicity": ["pos", "pos", "neg"],
    "beta_zero": [0.5, 0.5, 0.5],
    "beta_mode": ["arctan", "const", "const"],
    "alpha_zero": [0.0, 0.0, 0.0],
    "alpha_mode": ["const", "const", "const"],
    "RL_its": [0, 0, 0],
    "RL_freqs": [1e9, 1e9, 1e9],
    "TV_freqs": [1e9, 1e9, 1e9],
    "plot_every": [100, 50, 50],
    "average_img": [20, 20, 20],
    "Fourier_last": [True, True, True],
    "Startimage": [None, "pos", "pos"],
    "Startgamma": [None, None, None],
    "hologram_intensity_cutoff_vmin": -1,
}

In [ ]:
(
    retrieved_pos,
    retrieved_neg,
    retrieved_pos_pc,
    retrieved_neg_pc,
    bsmask_pos,
    bsmask_neg,
    gamma_pos,
    gamma_neg,
    error,
) = phr.phase_retrieval_algorithm(
    pos,
    neg,
    mask_pixel,
    supportmask,
    phase_retrieval_recipe=recipe,
)

phr.plot_phase_retrieval_errors(error, recipe)
plt.show()

## Inspect the support-domain objects

The returned fields use the library's Fourier-domain convention. Propagation and final physical scaling can be added later.

In [ ]:
object_pos = np.fft.fft2(np.fft.fftshift(retrieved_pos))
object_neg = np.fft.fft2(np.fft.fftshift(retrieved_neg))

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
axes[0, 0].imshow(np.abs(object_pos), cmap="gray")
axes[0, 0].set_title("Positive: amplitude")
axes[0, 1].imshow(np.angle(object_pos), cmap="twilight")
axes[0, 1].set_title("Positive: phase")
axes[1, 0].imshow(np.abs(object_neg), cmap="gray")
axes[1, 0].set_title("Negative: amplitude")
axes[1, 1].imshow(np.angle(object_neg), cmap="twilight")
axes[1, 1].set_title("Negative: phase")
for ax in axes.ravel():
    ax.axis("off")
plt.show()